## Setup and Imports

In [1]:
import sys
sys.path.insert(0, 'backend_functions')

import selection_functions as sf
import importlib
import uproot
import matplotlib.pylab as pylab
import numpy as np
import pandas as pd
import math
from sklearn.model_selection import train_test_split
import xgboost as xgb
import awkward
import matplotlib.pyplot as plt

# Top is where parameters(ISRUN3) is defined
import top 
from top import *

# Reload modules to pick up any changes
importlib.reload(sf)
from selection_functions import *

importlib.reload(top)
from top import *

print("✅ Imports complete")

✅ Imports complete


## Configuration

Set up which run periods to include in training and other parameters.

In [2]:
# ============================================================================
# HORN CURRENT CONFIGURATION
# ============================================================================
# Set horn current mode: 'FHC' (Forward Horn Current) or 'RHC' (Reverse Horn Current)
# This will automatically configure which run periods to use based on file availability:
#   - RHC mode: Uses Run 4a, 4b, 4c (RHC files only)
#   - FHC mode: Uses Run 4c, 4d, 5 (FHC files only)

HORN_CURRENT = 'RHC'  # Options: 'FHC' or 'RHC'

# Automatically set run periods based on horn current
if HORN_CURRENT == 'RHC':
    RUN_PERIODS = {
        'run4a': True,   # RHC only
        'run4b': True,   # RHC only + EXT
        'run4c': True,   # RHC files
        'run4d': False,  # No RHC data
        'run5': False    # No RHC data
    }
    ISRUN3 = True  # RHC mode
elif HORN_CURRENT == 'FHC':
    RUN_PERIODS = {
        'run4a': False,  # No FHC data
        'run4b': False,  # No FHC data
        'run4c': True,   # FHC files
        'run4d': True,   # FHC only
        'run5': True     # FHC only
    }
    ISRUN3 = False  # FHC mode
else:
    raise ValueError(f"Unknown HORN_CURRENT: {HORN_CURRENT}. Must be 'FHC' or 'RHC'")

# Other configuration
NUE_INTRINSIC = True
TRAIN_TEST_SPLIT = 0.5  # Use 50% for training, 50% for testing per run
USE_EXT_IN_BDT = False  # Whether to include EXT data in BDT training
EVENT_SPLIT_SEED = 17  # Seed for event-ID-based splitting

print("=" * 70)
print(f"🎯 HORN CURRENT MODE: {HORN_CURRENT}")
print("=" * 70)
print(f"Training configuration:")
print(f"  Run periods to include: {[k for k, v in RUN_PERIODS.items() if v]}")
print(f"  Train/Test split: {int((1-TRAIN_TEST_SPLIT)*100)}% train / {int(TRAIN_TEST_SPLIT*100)}% test")
print(f"  Use EXT in BDT: {USE_EXT_IN_BDT}")
print(f"  Event split seed: {EVENT_SPLIT_SEED}")
print(f"  ISRUN3 flag: {ISRUN3}")
print("=" * 70)


🎯 HORN CURRENT MODE: RHC
Training configuration:
  Run periods to include: ['run4a', 'run4b', 'run4c']
  Train/Test split: 50% train / 50% test
  Use EXT in BDT: False
  Event split seed: 17
  ISRUN3 flag: True


In [3]:
# Base path
base_path = "/pnfs/uboone/persistent/users/uboonepro/surprise/"

# ============================================================================
# FILE NAMING PATTERNS
# ============================================================================
# Define file patterns for each run period and horn current mode
# Based on actual file availability:
# - Run 4a: RHC only (overlay, nue, dirt), no EXT
# - Run 4b: RHC only (overlay, nue, dirt), EXT available
# - Run 4c: BOTH FHC and RHC (overlay, nue, dirt), no EXT
# - Run 4d: FHC only (overlay, nue, dirt), no EXT
# - Run 5:  FHC only (overlay, nue, dirt), no EXT

def get_file_paths(horn_current):
    """
    Get file paths for the specified horn current mode.
    
    Parameters:
    -----------
    horn_current : str
        'FHC' or 'RHC'
        
    Returns:
    --------
    dict : Dictionary of run periods with file paths
        Keys: run period names
        Values: dict with 'OVRLY', 'NUE', 'DIRT', and optionally 'EXT' keys
    """
    if horn_current == 'RHC':
        # RHC file paths
        return {
            'run4a': {
                'OVRLY': 'run4a_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_nu_overlay_surprise_reco2_hist_4a',
                'NUE': 'run4a_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_intrinsic_nue_overlay_surprise_reco2_hist_4a',
                'DIRT': 'run4a_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_dirt_overlay_surprise_reco2_hist_4a'
                # No EXT available for run4a yet
            },
            'run4b': {
                'OVRLY': 'run4b_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4b_NuMI_RHC_nu_overlay_surprise_v10_04_07_09_reco2_hist',
                'NUE': 'run4b_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4b_NuMI_RHC_nue_overlay_surprise_v10_04_07_09_reco2_hist',
                'DIRT': 'run4b_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_dirt_overlay_surprise_reco2_hist_4b',
                'EXT': 'run4b_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4b_NuMI_beam_off_RHC_data_surprise_v10_04_07_09_reco2_hist'  # Horn current independent
            },
            'run4c': {
                'OVRLY': 'run4c_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_nu_overlay_surprise_reco2_hist_4c',
                'NUE': 'run4c_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_intrinsic_nue_overlay_surprise_reco2_hist_4c',
                'DIRT': 'run4c_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4a4c_NuMI_RHC_dirt_overlay_surprise_reco2_hist_4c'
                # No EXT available for run4c yet
            }
            # Run 4d and Run 5 do not have RHC data
        }
    elif horn_current == 'FHC':
        # FHC file paths
        return {
            'run4c': {
                'OVRLY': 'run4c_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_nu_overlay_surprise_reco2_hist_4c',
                'NUE': 'run4c_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_intrinsic_nue_overlay_surprise_reco2_hist_4c',
                'DIRT': 'run4c_full_samples/wc_processed/NuMI/checkout_prodgenie_numi_fhc_dirt_new_flux_overlay_run_4cd_5_reco2_reco2_hist_4c'
                # No EXT available for run4c yet
            },
            'run4d': {
                'OVRLY': 'run4d_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_nu_overlay_surprise_reco2_hist_4d',
                'NUE': 'run4d_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_intrinsic_nue_overlay_surprise_reco2_hist_4d',
                'DIRT': 'run4d_full_samples/wc_processed/NuMI/checkout_prodgenie_numi_fhc_dirt_new_flux_overlay_run_4cd_5_reco2_reco2_hist_4d'
                # No EXT available for run4d yet
            },
            'run5': {
                'OVRLY': 'run5_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_nu_overlay_surprise_reco2_hist_5',
                'NUE': 'run5_full_samples/wc_processed/NuMI/checkout_MCC9.10_Run4c4d5_NuMI_FHC_intrinsic_nue_overlay_surprise_reco2_hist_5',
                'DIRT': 'run5_full_samples/wc_processed/NuMI/checkout_prodgenie_numi_fhc_dirt_new_flux_overlay_run_4cd_5_reco2_reco2_hist_5'
                # No EXT available for run5 yet
            }
            # Run 4a and Run 4b do not have FHC data
        }
    else:
        raise ValueError(f"Unknown horn current mode: {horn_current}. Must be 'FHC' or 'RHC'")


# Get file paths for the selected horn current
RUN_FILES = get_file_paths(HORN_CURRENT)

fold = "nuselection"
tree = "NeutrinoSelectionFilter"

print(f"\n✅ File paths configured for {HORN_CURRENT} mode")
print(f"   Available run periods: {list(RUN_FILES.keys())}")
print("⚠️  Note: Update FHC file paths in get_file_paths() function if needed")


✅ File paths configured for RHC mode
   Available run periods: ['run4a', 'run4b', 'run4c']
⚠️  Note: Update FHC file paths in get_file_paths() function if needed


In [4]:
# ============================================================================
# POT (Protons On Target) VALUES
# ============================================================================
# POT values for each run period - used for proper event weighting and normalization
# These are the beam-on data POT values

POT_VALUES = {
    'run4a': {
        'FHC': None,           # No FHC data for Run 4a
        'RHC': 8.312e+18       # 8.312E18
    },
    'run4b': {
        'FHC': None,           # No FHC data for Run 4b
        'RHC': 2.527e+20       # 2.527E20 (actual value, not scaled)
    },
    'run4c': {
        'FHC': 1.214e+20,      # 1.214E20
        'RHC': 1.688e+19       # 1.688E19
    },
    'run4d': {
        'FHC': 6.832e+19,      # 6.832E19
        'RHC': None            # No RHC data for Run 4d
    },
    'run5': {
        'FHC': 2.162e+20,      # 2.162E20
        'RHC': None            # No RHC data for Run 5
    }
}

# Get POT value for current horn current mode
def get_pot_for_run(run_name, horn_current):
    """Get the POT value for a specific run and horn current."""
    if run_name in POT_VALUES and POT_VALUES[run_name][horn_current] is not None:
        return POT_VALUES[run_name][horn_current]
    else:
        raise ValueError(f"No POT value available for {run_name} in {horn_current} mode")

# Display POT values for enabled runs
print("\n" + "=" * 70)
print("📊 POT VALUES FOR SELECTED RUNS")
print("=" * 70)
for run_name, enabled in RUN_PERIODS.items():
    if enabled and run_name in RUN_FILES:
        pot = POT_VALUES[run_name][HORN_CURRENT]
        print(f"  {run_name} ({HORN_CURRENT}): {pot:.3e}")
print("=" * 70)



📊 POT VALUES FOR SELECTED RUNS
  run4a (RHC): 8.312e+18
  run4b (RHC): 2.527e+20
  run4c (RHC): 1.688e+19


## Load Data from All Run Periods

Load files from each enabled run period.

In [5]:
def load_run_period(run_name, file_dict, include_nue=True):
    """
    Load all files for a single run period.
    ⚠️ DATA files excluded (blind analysis)
    
    Parameters:
    -----------
    run_name : str
        Name of the run period (e.g., 'run4a')
    file_dict : dict
        Dictionary containing file paths for this run
        Keys: 'OVRLY', 'NUE', 'DIRT', and optionally 'EXT'
    include_nue : bool
        Whether to include nue intrinsic sample
        
    Returns:
    --------
    list : List of uproot trees [overlay, ext (if available), nue (if requested), dirt]
    """
    print(f"\n📂 Loading {run_name}...")
    
    try:
        uproot_v = []
        file_types = []
        
        # Always load overlay
        overlay = uproot.open(base_path + file_dict['OVRLY'] + ".root")[fold][tree]
        uproot_v.append(overlay)
        file_types.append('overlay')
        
        # Load EXT if available and requested
        if 'EXT' in file_dict and USE_EXT_IN_BDT:
            ext = uproot.open(base_path + file_dict['EXT'] + ".root")[fold][tree]
            uproot_v.append(ext)
            file_types.append('ext')
        elif 'EXT' not in file_dict:
            print(f"   ⚠️  No EXT file available for {run_name}")
        
        # Load NUE if requested
        if include_nue and 'NUE' in file_dict:
            nue = uproot.open(base_path + file_dict['NUE'] + ".root")[fold][tree]
            uproot_v.append(nue)
            file_types.append('nue')
            
        # Load DIRT if available
        if 'DIRT' in file_dict:
            dirt = uproot.open(base_path + file_dict['DIRT'] + ".root")[fold][tree]
            uproot_v.append(dirt)
            file_types.append('dirt')
        else:
            print(f"   ⚠️  No DIRT file available for {run_name}")
            
        print(f"   ✅ Loaded: {', '.join(file_types)}")
        print(f"   ⚠️  DATA files excluded (blind analysis)")
        return uproot_v
        
    except Exception as e:
        print(f"   ❌ Error loading {run_name}: {e}")
        return None


# Load all enabled run periods
all_run_data = {}

for run_name, enabled in RUN_PERIODS.items():
    if enabled:
        # Check if this run exists in the file paths
        if run_name in RUN_FILES:
            run_data = load_run_period(run_name, RUN_FILES[run_name], include_nue=NUE_INTRINSIC)
            if run_data is not None:
                all_run_data[run_name] = run_data
        else:
            print(f"\n⚠️  Run {run_name} not available for {HORN_CURRENT} mode")

print(f"\n✅ Successfully loaded {len(all_run_data)} run period(s)")


📂 Loading run4a...
   ⚠️  No EXT file available for run4a
   ✅ Loaded: overlay, nue, dirt
   ⚠️  DATA files excluded (blind analysis)

📂 Loading run4b...
   ✅ Loaded: overlay, nue, dirt
   ⚠️  DATA files excluded (blind analysis)

📂 Loading run4c...
   ⚠️  No EXT file available for run4c
   ✅ Loaded: overlay, nue, dirt
   ⚠️  DATA files excluded (blind analysis)

✅ Successfully loaded 3 run period(s)


In [6]:
# Track file types for each run period
# This is needed to know which files were loaded for each run
all_file_types = {}

for run_name in all_run_data.keys():
    file_types = ['overlay']  # Always have overlay
    
    if 'EXT' in RUN_FILES[run_name] and USE_EXT_IN_BDT:
        file_types.append('ext')
    
    if NUE_INTRINSIC and 'NUE' in RUN_FILES[run_name]:
        file_types.append('nue')
    
    if 'DIRT' in RUN_FILES[run_name]:
        file_types.append('dirt')
    
    all_file_types[run_name] = file_types

print(f"\n📋 File types loaded for each run:")
for run_name, ftypes in all_file_types.items():
    print(f"  {run_name}: {', '.join(ftypes)}")


📋 File types loaded for each run:
  run4a: overlay, nue, dirt
  run4b: overlay, nue, dirt
  run4c: overlay, nue, dirt


## Load Variables and Apply Preselection

Load the necessary variables for BDT training from each run period.

In [7]:
# Get variable list from top.py and add required variables
# Need to load raw variables from ROOT files before computing derived ones

# Variables needed from ROOT files (not yet computed)
root_variables = [
    'run', 'sub', 'evt',  # Event IDs
    'swtrig_pre',  # Software trigger
    'slice_orig_pass_id',  # Slice selection
    # Reco vertex
    'reco_nu_vtx_sce_x', 'reco_nu_vtx_sce_y', 'reco_nu_vtx_sce_z',
    # Truth info (MC only)
    'nu_pdg', 'ccnc', 'nu_e',
    'true_nu_vtx_x', 'true_nu_vtx_y', 'true_nu_vtx_z',
    'npi0', 'npion', 'nproton', 'elec_e',
    # Reco variables
    'n_tracks_contained', 'n_showers_contained',
    'trk_energy', 'shr_energy_tot_cali', 'shr_energy_cali',
    'contained_fraction', 'shrmoliereavg',
    'tksh_distance', 'tksh_angle', 'trkshrhitdist2',
    'NeutrinoEnergy2',  # Needed for NeutrinoEnergy2_GeV calculation
    # dE/dx variables
    'shr_tkfit_dedx_Y', 'shr_tkfit_gap10_dedx_Y', 'shr_tkfit_2cm_dedx_Y',
    # Subcluster components (need these to compute 'subcluster')
    'shrsubclusters0', 'shrsubclusters1', 'shrsubclusters2',
    # NuGraph variables (raw from file)
    'ng2hip_r1cm', 'ng2hip_r10cm',
    # Weight variables (MC only)
    'weightSpline', 'weightSplineTimesTune', 'weightTune', 'ppfx_cv',
]

# Variables that will be computed after loading (trkpid, pfng2*, subcluster, pot_scale, totweight_data)
# These are NOT in the ROOT files and must be computed

# Remove duplicates
variables = list(set(root_variables))

print(f"Loading {len(variables)} raw variables from ROOT files...")
print(f"First 10: {variables[:10]}")
print(f"\n⚠️  Note: Derived variables (trkpid, pfng2*, subcluster) will be computed after loading")

Loading 41 raw variables from ROOT files...
First 10: ['n_showers_contained', 'sub', 'shrsubclusters2', 'ppfx_cv', 'shrsubclusters0', 'reco_nu_vtx_sce_z', 'weightSpline', 'ccnc', 'elec_e', 'contained_fraction']

⚠️  Note: Derived variables (trkpid, pfng2*, subcluster) will be computed after loading


In [ ]:
def load_variables_from_run(uproot_trees, variables, file_types):
    """
    Load variables from uproot trees for a run period.
    
    Parameters:
    -----------
    uproot_trees : list
        List of uproot trees (overlay, ext if available, nue if requested, dirt)
    variables : list
        List of variable names to load
    file_types : list
        Names of the file types corresponding to uproot_trees
        
    Returns:
    --------
    tuple : (datasets, file_types)
        datasets: List of DataFrames
        file_types: List of corresponding type names
    """
    datasets = []
    
    for i, tree in enumerate(uproot_trees):
        try:
            # Load variables that exist in the tree
            available_vars = [v for v in variables if v in tree.keys()]
            df = tree.arrays(available_vars, library="pd")
            datasets.append(df)
            print(f"   ✅ {file_types[i]}: {len(df):,} events, {len(available_vars)} variables")
        except Exception as e:
            print(f"   ❌ Error loading {file_types[i]}: {e}")
            
    return datasets, file_types


# Load variables from all run periods
all_datasets = {}

for run_name, uproot_trees in all_run_data.items():
    print(f"\n📊 Loading variables for {run_name}:")
    file_types = all_file_types[run_name]
    datasets, _ = load_variables_from_run(uproot_trees, variables, file_types)
    all_datasets[run_name] = datasets


print(f"\n✅ Loaded variables from {len(all_datasets)} run period(s)")


📊 Loading variables for run4a:


## Compute Derived Variables

Compute NuGraph variables, trkpid, and other derived variables that are needed for BDT training (same as selection.ipynb).

In [ ]:
# Define functions to compute NuGraph variables and trkpid from uproot trees
# These are the same functions used in selection.ipynb

def compute_trkpid(up):
    """Compute track PID score from LLR PID scores"""
    arrs = up.arrays(['trk_llr_pid_score_v', 'trk_id'], library='np')
    pid_v = arrs['trk_llr_pid_score_v']
    tid = arrs['trk_id'].astype('int64') - 1  # convert to 0-based
    out = []
    for pid, t in zip(pid_v, tid):
        ti = int(t)
        if 0 <= ti < len(pid):
            out.append(float(pid[ti]))
        else:
            out.append(9999.0)
    return out

def compute_pfng2hipavrg(up):
    """NuGraph HIP average for track"""
    arrs = up.arrays(['pfng2hipavrg', 'trk_id'], library='np')
    sem_v = arrs['pfng2hipavrg']
    tid = arrs['trk_id'].astype('int64') - 1
    out = []
    for sem, t in zip(sem_v, tid):
        ti = int(t)
        if 0 <= ti < len(sem):
            out.append(float(sem[ti]))
        else:
            out.append(9999.0)
    return out

def compute_pfng2shravrg(up):
    """NuGraph shower average for shower"""
    arrs = up.arrays(['pfng2shravrg', 'shr_id'], library='np')
    sem_v = arrs['pfng2shravrg']
    tid = arrs['shr_id'].astype('int64') - 1
    out = []
    for sem, t in zip(sem_v, tid):
        ti = int(t)
        if 0 <= ti < len(sem):
            out.append(float(sem[ti]))
        else:
            out.append(9999.0)
    return out

def compute_pfng2hipfrac(up):
    """NuGraph HIP fraction for track"""
    arrs = up.arrays(['pfng2hipfrac', 'trk_id'], library='np')
    sem_v = arrs['pfng2hipfrac']
    tid = arrs['trk_id'].astype('int64') - 1
    out = []
    for sem, t in zip(sem_v, tid):
        ti = int(t)
        if 0 <= ti < len(sem):
            out.append(float(sem[ti]))
        else:
            out.append(9999.0)
    return out

def compute_pfng2shrfrac(up):
    """NuGraph shower fraction for shower"""
    arrs = up.arrays(['pfng2shrfrac', 'shr_id'], library='np')
    sem_v = arrs['pfng2shrfrac']
    tid = arrs['shr_id'].astype('int64') - 1
    out = []
    for sem, t in zip(sem_v, tid):
        ti = int(t)
        if 0 <= ti < len(sem):
            out.append(float(sem[ti]))
        else:
            out.append(9999.0)
    return out

print("✅ Variable computation functions defined")

In [ ]:
# Compute NuGraph variables and trkpid for each run period
# Apply to overlay and nue (not EXT - no MC truth info)

print("=" * 70)
print("🔬 COMPUTING DERIVED VARIABLES")
print("=" * 70)
print()

for run_name in all_run_data.keys():
    print(f"📊 Computing variables for {run_name}...")
    
    uproot_trees = all_run_data[run_name]
    datasets = all_datasets[run_name]
    file_types = all_file_types[run_name]
    
    for i, (df, uproot_tree, file_type) in enumerate(zip(datasets, uproot_trees, file_types)):
        # Only compute for MC samples (overlay, nue, dirt) - not EXT
        if file_type in ['overlay', 'nue', 'dirt']:
            print(f"   Computing for {file_type}...")
            
            # Compute NuGraph and trkpid variables
            df['trkpid'] = compute_trkpid(uproot_tree)
            df['pfng2hipavrg'] = compute_pfng2hipavrg(uproot_tree)
            df['pfng2shravrg'] = compute_pfng2shravrg(uproot_tree)
            df['pfng2hipfrac'] = compute_pfng2hipfrac(uproot_tree)
            df['pfng2shrfrac'] = compute_pfng2shrfrac(uproot_tree)
            
            # Compute subcluster (sum of subclusters)
            df['subcluster'] = df['shrsubclusters0'] + df['shrsubclusters1'] + df['shrsubclusters2']
            
            print(f"      ✅ Added: trkpid, pfng2hipavrg, pfng2shravrg, pfng2hipfrac, pfng2shrfrac, subcluster")
        
        # Update the dataset in all_datasets
        all_datasets[run_name][i] = df
    
    print()

print("=" * 70)
print("✅ Derived variable computation complete!")
print("=" * 70)

## Apply Common Derived Columns

Apply energy calibrations and other common derived columns that are applied to ALL samples in selection.ipynb (including EXT).

In [ ]:
# Apply common derived columns to ALL samples (MC and EXT)
# Following selection.ipynb lines 450-456

print("=" * 70)
print("🔬 APPLYING COMMON DERIVED COLUMNS")
print("=" * 70)
print()

for run_name in all_datasets.keys():
    print(f"📊 Processing {run_name}...")
    
    datasets = all_datasets[run_name]
    file_types = all_file_types[run_name]
    
    for i, (df, file_type) in enumerate(zip(datasets, file_types)):
        # Apply to ALL samples (overlay, ext, nue, dirt, data if present)
        print(f"   Applying to {file_type}...")
        
        # Energy calibrations
        df['shr_energy_cali'] = df['shr_energy_cali'] / 0.83
        df['NeutrinoEnergy2_GeV'] = df['NeutrinoEnergy2'] / 1000.0
        
        print(f"      ✅ Applied: shr_energy_cali/0.83, NeutrinoEnergy2_GeV")
        
        # Update the dataset
        all_datasets[run_name][i] = df
    
    print()

print("=" * 70)
print("✅ Common derived columns applied!")
print("=" * 70)

## Clean Weights and Add is_signal

Clean bad weight values and add the is_signal flag (same as selection.ipynb).

In [ ]:
# Clean bad weights and add is_signal flag for MC samples
# Following selection.ipynb approach

print("=" * 70)
print("🧹 CLEANING WEIGHTS AND ADDING is_signal FLAG")
print("=" * 70)
print()

for run_name in all_datasets.keys():
    print(f"📊 Processing {run_name}...")
    
    datasets = all_datasets[run_name]
    file_types = all_file_types[run_name]
    
    for i, (df, file_type) in enumerate(zip(datasets, file_types)):
        # Only for MC samples (overlay, nue, dirt)
        if file_type in ['overlay', 'nue', 'dirt']:
            print(f"   Cleaning {file_type}...")
            
            # Clean bad weights (same as selection.ipynb)
            for weight_var in ['ppfx_cv', 'weightSplineTimesTune', 'weightTune']:
                if weight_var in df.columns:
                    df.loc[df[weight_var] <= 0, weight_var] = 1.0
                    df.loc[df[weight_var] == np.inf, weight_var] = 1.0
                    df.loc[df[weight_var] > 30, weight_var] = 1.0
                    df.loc[np.isnan(df[weight_var]), weight_var] = 1.0
            
            # Add is_signal flag (same definition as selection.ipynb)
            df['is_signal'] = np.where(
                (df['nu_pdg'] == 12) & 
                (df['ccnc'] == 0) & 
                (df['nproton'] > 0) & 
                (df['npion'] == 0) & 
                (df['npi0'] == 0) &
                (10 <= df['true_nu_vtx_x']) & (df['true_nu_vtx_x'] <= 246) &
                (-106 <= df['true_nu_vtx_y']) & (df['true_nu_vtx_y'] <= 106) &
                (10 <= df['true_nu_vtx_z']) & (df['true_nu_vtx_z'] <= 1026) &
                (df['elec_e'] > 0.07),  # Electron energy threshold
                True, False
            )
            
            n_signal = df['is_signal'].sum()
            print(f"      ✅ Cleaned weights, added is_signal ({n_signal:,} signal events)")
        
        # Update dataset
        all_datasets[run_name][i] = df
    
    print()

print("=" * 70)
print("✅ Weight cleaning and is_signal complete!")
print("=" * 70)

## MC POT Values

Each MC sample (overlay, nue intrinsic, dirt) was generated with its own POT. 
We need these values to calculate `pot_scale` for proper normalization.

In [ ]:
# ============================================================================
# MC GENERATION POT VALUES
# ============================================================================
# POT values used when generating each MC sample type
# These are needed to calculate pot_scale = beamon_pot / mc_generation_pot

MC_GENERATION_POT = {
    'run4a': {
        'RHC': {
            'overlay': 7.20478e+20,
            'nue': 1.49847e+22,
            'dirt': 1.3077e+20
        },
        'FHC': {
            'overlay': None,      # No FHC for Run 4a
            'nue': None,
            'dirt': None
        }
    },
    'run4b': {
        'RHC': {
            'overlay': 2.33807e+21,
            'nue': 5.04447e+22,
            'dirt': 4.20894e+20
        },
        'FHC': {
            'overlay': None,      # No FHC for Run 4b
            'nue': None,
            'dirt': None
        }
    },
    'run4c': {
        'RHC': {
            'overlay': 1.47086e+20,
            'nue': 3.07874e+21,
            'dirt': 2.60777e+19
        },
        'FHC': {
            'overlay': 1.05787e+21,
            'nue': 2.36381e+22,
            'dirt': 2.15437e+20
        }
    },
    'run4d': {
        'RHC': {
            'overlay': None,      # No RHC for Run 4d
            'nue': None,
            'dirt': None
        },
        'FHC': {
            'overlay': 1.69211e+21,
            'nue': 4.95459e+22,
            'dirt': 3.4002e+20
        }
    },
    'run5': {
        'RHC': {
            'overlay': None,      # No RHC for Run 5
            'nue': None,
            'dirt': None
        },
        'FHC': {
            'overlay': 2.61871e+21,
            'nue': 6.01751e+22,
            'dirt': 4.95156e+20
        }
    }
}

print("=" * 70)
print("📋 MC GENERATION POT VALUES NEEDED")
print("=" * 70)
print("\n⚠️  TODO: Fill in MC generation POT values for enabled runs")
print("\nFor each enabled run, you need:")
print("  - Overlay generation POT")
print("  - Nue intrinsic generation POT")
print("  - Dirt generation POT (if using dirt)")
print("\nThese values are typically found in:")
print("  - SAM metadata for the MC samples")
print("  - Production documentation")
print("  - metadata.txt files in the sample directories")
print("=" * 70)

## Calculate pot_scale for Each Dataset

Now calculate `pot_scale` for each MC sample: `pot_scale = beamon_pot / mc_generation_pot`

In [ ]:
def calculate_pot_scale(datasets, run_name, horn_current, file_types):
    """
    Calculate and add pot_scale column to each dataset.
    
    pot_scale = beamon_pot / mc_generation_pot
    
    This normalizes each MC sample to the beam-on data POT for that run.
    
    Parameters:
    -----------
    datasets : list of DataFrames
        List of datasets for this run (overlay, ext, nue, dirt)
    run_name : str
        Name of the run period (e.g., 'run4b')
    horn_current : str
        'FHC' or 'RHC'
    file_types : list
        Names of file types corresponding to datasets
        
    Returns:
    --------
    list of DataFrames : Datasets with pot_scale column added
    """
    # Get beam-on POT for this run
    beamon_pot = POT_VALUES[run_name][horn_current]
    
    print(f"   {run_name} ({horn_current}):")
    print(f"      Beam-on POT: {beamon_pot:.3e}")
    
    scaled_datasets = []
    
    for i, df in enumerate(datasets):
        df_scaled = df.copy()
        file_type = file_types[i]
        
        # Skip EXT - it doesn't use POT scaling (uses trigger ratio instead)
        if file_type == 'ext':
            # For EXT, we would need trigger counts instead
            # For now, set pot_scale = 1 (will need proper EXT scaling later)
            df_scaled['pot_scale'] = 1.0
            print(f"      {file_type}: pot_scale set to 1.0 (needs trigger ratio)")
            scaled_datasets.append(df_scaled)
            continue
        
        # For MC samples (overlay, nue, dirt)
        mc_gen_pot = MC_GENERATION_POT[run_name][horn_current].get(file_type)
        
        if mc_gen_pot is None:
            raise ValueError(
                f"Missing MC generation POT for {run_name} {horn_current} {file_type}. "
                f"Please fill in MC_GENERATION_POT dictionary above."
            )
        
        # Calculate pot_scale
        pot_scale = beamon_pot / mc_gen_pot
        df_scaled['pot_scale'] = pot_scale
        
        print(f"      {file_type}: MC gen POT = {mc_gen_pot:.3e}, pot_scale = {pot_scale:.6f}")
        
        scaled_datasets.append(df_scaled)
    
    return scaled_datasets


# Apply pot_scale calculation to all datasets
print("=" * 70)
print("🔬 CALCULATING POT_SCALE FOR EACH DATASET")
print("=" * 70)
print()

all_datasets_with_pot_scale = {}

for run_name, datasets in all_datasets.items():
    file_types = all_file_types[run_name]
    
    try:
        scaled_datasets = calculate_pot_scale(
            datasets, 
            run_name, 
            HORN_CURRENT, 
            file_types
        )
        all_datasets_with_pot_scale[run_name] = scaled_datasets
        print()
        
    except ValueError as e:
        print(f"\n❌ Error: {e}")
        print("\nPlease fill in the MC_GENERATION_POT values and rerun this cell.")
        raise

# Replace datasets with pot_scale versions
all_datasets = all_datasets_with_pot_scale

print("=" * 70)
print("✅ pot_scale calculation complete!")
print("=" * 70)

## Calculate totweight_data

Calculate the combined weight that will be used in BDT training:
`totweight_data = pot_scale * ppfx_cv * weightSplineTimesTune`

This is the actual weight variable that will be used for MC samples in the BDT.

In [ ]:
# Calculate totweight_data for all MC samples
# This is the weight used in BDT training: pot_scale * ppfx_cv * weightSplineTimesTune

print("=" * 70)
print("🔬 CALCULATING totweight_data FOR BDT TRAINING")
print("=" * 70)
print()

for run_name in all_datasets.keys():
    print(f"📊 Processing {run_name}...")
    
    datasets = all_datasets[run_name]
    file_types = all_file_types[run_name]
    
    for i, (df, file_type) in enumerate(zip(datasets, file_types)):
        # For MC samples (overlay, nue, dirt): totweight_data = pot_scale * ppfx_cv * weightSplineTimesTune
        if file_type in ['overlay', 'nue', 'dirt']:
            df['totweight_data'] = df['pot_scale'] * df['ppfx_cv'] * df['weightSplineTimesTune']
            print(f"   {file_type}: totweight_data = pot_scale * ppfx_cv * weightSplineTimesTune")
        
        # For EXT and DATA: totweight_data = NaN (not used in training weight)
        elif file_type in ['ext', 'data']:
            df['totweight_data'] = np.nan
            print(f"   {file_type}: totweight_data = NaN (not used)")
        
        # Update dataset
        all_datasets[run_name][i] = df
    
    print()

print("=" * 70)
print("✅ totweight_data calculation complete!")
print("=" * 70)
print("\nThis weight will be used in addRelevantColumns_flexible() to set the")
print("'weight' column for BDT training (combines POT, flux, and tune weights).")

## Multi-Run POT Normalization

Now that each dataset has `pot_scale` calculated (normalizing MC to beamon_pot within each run), 
we need to scale **between runs** so each run contributes proportionally to its total beam exposure.

This is a second layer of normalization:
1. **Sample-level** (just completed): overlay/nue/dirt → normalized to run's beamon_pot
2. **Run-level** (next): scale between runs based on relative POT

In [ ]:
# ============================================================================
# MULTI-RUN POT NORMALIZATION
# ============================================================================

def renormalize_pot_weights(datasets, run_name, horn_current, total_pot):
    """
    Re-normalize POT weights for multi-run training.
    
    This adjusts weights so that each run contributes proportionally to its 
    actual beam exposure relative to the total combined POT.
    
    Parameters:
    -----------
    datasets : list of DataFrames
        List of datasets for this run (overlay, ext, nue, dirt)
    run_name : str
        Name of the run period
    horn_current : str
        'FHC' or 'RHC'
    total_pot : float
        Total POT across all enabled runs
        
    Returns:
    --------
    list of DataFrames : Datasets with adjusted weights
    """
    # Get actual POT for this run
    run_pot = POT_VALUES[run_name][horn_current]
    
    # Calculate adjustment factor:
    # Normalize each run to the total combined POT so that each run's 
    # contribution is proportional to its actual beam exposure
    adjustment = total_pot / run_pot
    
    print(f"   {run_name}:")
    print(f"      POT: {run_pot:.3e}")
    print(f"      Contribution: {run_pot/total_pot*100:.1f}%")
    print(f"      Adjustment factor: {adjustment:.4f}")
    
    # Apply adjustment to all datasets for this run
    adjusted_datasets = []
    for df in datasets:
        df_adjusted = df.copy()
        
        # Adjust weight columns if they exist
        if 'pot_scale' in df_adjusted.columns:
            df_adjusted['pot_scale'] *= adjustment
        
        if 'totweight_data' in df_adjusted.columns:
            df_adjusted['totweight_data'] *= adjustment
            
        adjusted_datasets.append(df_adjusted)
    
    return adjusted_datasets


# Calculate total POT across all enabled runs
print("=" * 70)
print("🔬 CALCULATING MULTI-RUN POT NORMALIZATION")
print("=" * 70)

enabled_runs = [run for run, enabled in RUN_PERIODS.items() if enabled and run in RUN_FILES]
total_pot = sum(POT_VALUES[run][HORN_CURRENT] for run in enabled_runs)

print(f"\nTotal POT across all enabled runs: {total_pot:.3e}")
print(f"Using total POT as common reference for all runs")
print(f"\nRe-normalizing weights for each run:\n")

# Apply renormalization to all datasets
all_datasets_normalized = {}

for run_name, datasets in all_datasets.items():
    normalized_datasets = renormalize_pot_weights(
        datasets, 
        run_name, 
        HORN_CURRENT, 
        total_pot
    )
    all_datasets_normalized[run_name] = normalized_datasets

# Replace original datasets with normalized versions
all_datasets = all_datasets_normalized

print("\n" + "=" * 70)
print("✅ POT normalization complete!")
print("=" * 70)
print("\nEach run now contributes to training proportionally to its beam exposure.")
print("This ensures proper statistical weighting in the multi-run BDT model.")

In [ ]:
def split_datasets(datasets, file_types, test_fraction=0.5, seed=17):
    """
    Apply sklearn train_test_split to all datasets in a run period.
    Uses stratification on signal/background when possible.
    Note: DATA files excluded (blind analysis).
    
    Parameters:
    -----------
    datasets : list
        List of DataFrames (can include overlay, ext, nue, dirt)
    file_types : list
        Names of file types corresponding to datasets
    test_fraction : float
        Fraction for test set
    seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    train_datasets, test_datasets : tuple of lists
        Split datasets
    """
    from sklearn.model_selection import train_test_split
    
    train_datasets = []
    test_datasets = []
    
    for i, df in enumerate(datasets):
        if len(df) > 0:
            # Stratify on the exact is_signal definition if available (MC only)
            # This ensures signal events are balanced equally between train and test.
            stratify_col = None
            if file_types[i] in ['overlay', 'nue'] and 'is_signal' in df.columns:
                stratify_col = df['is_signal']
            
            try:
                train_df, test_df = train_test_split(
                    df, 
                    test_size=test_fraction, 
                    random_state=seed,
                    stratify=stratify_col
                )
                train_datasets.append(train_df)
                test_datasets.append(test_df)
                print(f"   {file_types[i]}: {len(train_df):,} train, {len(test_df):,} test")
                
            except ValueError as e:
                # If stratification fails (e.g. too few signal events), split without it
                print(f"   Warning: Could not stratify {file_types[i]}, splitting without stratification")
                train_df, test_df = train_test_split(
                    df, 
                    test_size=test_fraction, 
                    random_state=seed
                )
                train_datasets.append(train_df)
                test_datasets.append(test_df)
                print(f"   {file_types[i]}: {len(train_df):,} train, {len(test_df):,} test")
        else:
            train_datasets.append(df)
            test_datasets.append(df)
            
    return train_datasets, test_datasets


# Split each run period independently with the same random_state
train_data_by_run = {}
test_data_by_run = {}

print("Splitting each run period using train_test_split...\n")

for run_name, datasets in all_datasets.items():
    print(f"🔀 Splitting {run_name}:")
    file_types = all_file_types[run_name]
    train_sets, test_sets = split_datasets(datasets, file_types, TRAIN_TEST_SPLIT, EVENT_SPLIT_SEED)
    train_data_by_run[run_name] = train_sets
    test_data_by_run[run_name] = test_sets

print(f"\n✅ Split complete for all run periods")
print(f"   Each run was split independently with random_state={EVENT_SPLIT_SEED}")

## Combine and Categorize Training Data

Combine MC from all runs and split into infv (in fiducial volume) and outfv (out of fiducial volume) based on **truth vertex** location, just like selection.ipynb.

In [ ]:
# Combine all MC from all run periods (overlay + nue if using intrinsic)
# Following selection.ipynb approach

print("=" * 70)
print("🔬 COMBINING AND CATEGORIZING MC DATA")
print("=" * 70)
print()

# Step 1: Combine overlay from all runs
print("📦 Combining overlay samples from all run periods...")
train_overlay_combined = []
test_overlay_combined = []

for run_name in all_datasets.keys():
    # Index 0 is always overlay
    train_overlay_combined.append(train_data_by_run[run_name][0])
    test_overlay_combined.append(test_data_by_run[run_name][0])

train_overlay = pd.concat(train_overlay_combined, ignore_index=True)
test_overlay = pd.concat(test_overlay_combined, ignore_index=True)

print(f"   Train overlay: {len(train_overlay):,} events")
print(f"   Test overlay: {len(test_overlay):,} events")

# Step 2: If using nue intrinsic, combine nue and replace nueCC events in overlay
if NUE_INTRINSIC:
    print("\n📦 Combining nue intrinsic samples...")
    
    # Combine nue from all runs
    train_nue_combined = []
    test_nue_combined = []
    
    for run_name in all_datasets.keys():
        file_types = all_file_types[run_name]
        nue_index = file_types.index('nue') if 'nue' in file_types else None
        if nue_index is not None:
            train_nue_combined.append(train_data_by_run[run_name][nue_index])
            test_nue_combined.append(test_data_by_run[run_name][nue_index])
    
    train_nue = pd.concat(train_nue_combined, ignore_index=True)
    test_nue = pd.concat(test_nue_combined, ignore_index=True)
    
    print(f"   Train nue: {len(train_nue):,} events")
    print(f"   Test nue: {len(test_nue):,} events")
    
    # Replace nueCC events in overlay with intrinsic sample (same as selection.ipynb)
    print("\n🔄 Replacing nueCC in AV events in overlay with intrinsic sample...")
    
    # Define nueCC query (from top.py)
    in_AV_query = "-1.55<=true_nu_vtx_x<=254.8 and -116.5<=true_nu_vtx_y<=116.5 and 0<=true_nu_vtx_z<=1036.8"
    nueCC_query = in_AV_query + ' and ((nu_pdg==12 or nu_pdg==-12) and ccnc==0)'
    
    # Train set
    len1_train = len(train_overlay)
    idx_train = train_overlay.query(nueCC_query).index
    train_overlay.drop(idx_train, inplace=True)
    len2_train = len(train_overlay)
    print(f"   Train: Dropped {len1_train - len2_train:,} nueCC in AV events from overlay")
    
    train_overlay = pd.concat([train_overlay, train_nue], ignore_index=True)
    print(f"   Train: Added {len(train_nue):,} intrinsic nue events")
    
    # Test set
    len1_test = len(test_overlay)
    idx_test = test_overlay.query(nueCC_query).index
    test_overlay.drop(idx_test, inplace=True)
    len2_test = len(test_overlay)
    print(f"   Test: Dropped {len1_test - len2_test:,} nueCC in AV events from overlay")
    
    test_overlay = pd.concat([test_overlay, test_nue], ignore_index=True)
    print(f"   Test: Added {len(test_nue):,} intrinsic nue events")

# Step 3: Split combined MC into infv (in FV) and outfv (out of FV) based on truth vertex
print("\n🎯 Splitting MC into infv (in FV) and outfv (out of FV)...")
print("   Using truth vertex location (from top.py):")
print(f"   in_fv_query: {in_fv_query}")
print(f"   out_fv_query: {out_fv_query}")

# Train sets
train_infv = train_overlay.query(in_fv_query)
train_outfv = train_overlay.query(out_fv_query)

print(f"\n   Train infv: {len(train_infv):,} events")
print(f"   Train outfv: {len(train_outfv):,} events")
print(f"   Total: {len(train_infv) + len(train_outfv):,} (overlay: {len(train_overlay):,})")

# Verify all events are accounted for
if len(train_overlay) != len(train_infv) + len(train_outfv):
    print(f"   ⚠️  WARNING: {len(train_overlay) - len(train_infv) - len(train_outfv)} events unaccounted for!")

# Test sets
test_infv = test_overlay.query(in_fv_query)
test_outfv = test_overlay.query(out_fv_query)

print(f"\n   Test infv: {len(test_infv):,} events")
print(f"   Test outfv: {len(test_outfv):,} events")
print(f"   Total: {len(test_infv) + len(test_outfv):,} (overlay: {len(test_overlay):,})")

if len(test_overlay) != len(test_infv) + len(test_outfv):
    print(f"   ⚠️  WARNING: {len(test_overlay) - len(test_infv) - len(test_outfv)} events unaccounted for!")

print("\n" + "=" * 70)
print("✅ MC categorization complete!")
print("=" * 70)
print("\n📊 Summary:")
print(f"   Training: {len(train_infv):,} infv + {len(train_outfv):,} outfv = {len(train_infv) + len(train_outfv):,} total")
print(f"   Testing: {len(test_infv):,} infv + {len(test_outfv):,} outfv = {len(test_infv) + len(test_outfv):,} total")
print(f"   EXT: NOT used in BDT training (USE_EXT_IN_BDT={USE_EXT_IN_BDT})")
print("=" * 70)

## BDT Training - Step 1: Optimize Number of Rounds

First, we optimize the number of boosting rounds using AUC and AUCPR metrics with early stopping.
This follows the same approach as in selection.ipynb.

## BDT Training Workflow

Following the same approach as `selection.ipynb`:

**Step 1: Optimize Rounds** (next cell)
- Use `bdt_metrics()` with early stopping (up to 1000 rounds, stops after 50 rounds without improvement)
- Plot AUC and AUCPR curves
- Determine optimal number of boosting rounds

**Step 2: Train Final Model** (cells below)
- Set `ROUNDS` based on Step 1 results
- Train model with `sf.bdt_raw_results()` using the pre-split DataFrames and shared `params`
- Evaluate performance and save model

In [ ]:
# BDT training queries
# TRAIN ON A SUBSET OF THE DISTRIBUTION (same as selection.ipynb)
TRAIN_QUERY = BDT_LOOSE_CUTS + ' and -0.9<tksh_angle<0.9'  # Train on subset
TEST_QUERY = BDT_LOOSE_CUTS  # Test on full distribution after loose cuts

print("🎯 BDT Training Configuration:")
print(f"   Training query: {TRAIN_QUERY}")
print(f"   Test query: {TEST_QUERY}")
print(f"   Training variables: {training_parameters}")
print(f"   Use EXT in BDT: {USE_EXT_IN_BDT}")
print(f"   Train/Test split: {int((1-TRAIN_TEST_SPLIT)*100)}% train / {int(TRAIN_TEST_SPLIT*100)}% test")

In [ ]:
# Prepare training data: build df_pre_train and df_pre_test from the already-split
# infv/outfv sets (split_datasets already did the 50/50 split in cell 28).
# No second train_test_split is needed here.

df_pre_train = sf.addRelevantColumns_flexible(
    {'infv': train_infv, 'outfv': train_outfv}, USE_EXT_IN_BDT=USE_EXT_IN_BDT
)
df_pre_test = sf.addRelevantColumns_flexible(
    {'infv': test_infv, 'outfv': test_outfv}, USE_EXT_IN_BDT=USE_EXT_IN_BDT
)

# Full combined set — used for scale_pos_weight calculation later
df_pre = pd.concat([df_pre_train, df_pre_test], ignore_index=True)

print(f"📊 Training set: {len(df_pre_train):,} events")
print(f"📊 Test set:     {len(df_pre_test):,} events")
print(f"   Signal in training:    {df_pre_train['is_signal'].sum():,}")
print(f"   Background in training:   {(~df_pre_train['is_signal']).sum():,}")


## Clean Training Features

Replace sentinel values (|x| ≥ 9999), ±inf, and float32 overflows with `NaN` in the BDT feature columns.  
`NaN` is passed directly to XGBoost, which learns the optimal branch direction for missing values — this is more informative than imputing a median.

In [ ]:
SENTINEL_THRESHOLD = 9999        # |x| >= this → failed reconstruction
float32_max = np.finfo(np.float32).max   # ~3.4e38

# Features that use -1 as an explicit "reco failed" sentinel.
# Only variables where -1 is physically impossible (distances, not angles).
SENTINEL_MINUS1_FEATURES = ['trkshrhitdist2']

def replace_bad_with_nan(df, features):
    """
    Replace sentinel values, ±inf, and float32 overflows with NaN
    in the given feature columns, in-place.
    Returns a diagnostic DataFrame listing affected columns.
    
    Sentinels handled:
      - ±inf
      - |x| > 0.1 * float32_max  (~3.4e37)  — float32 overflow/underflow
      - |x| >= SENTINEL_THRESHOLD (9999)     — explicit reco-fail sentinels & INT_MIN/MAX
      - x == -1 for SENTINEL_MINUS1_FEATURES — trkshrhitdist2 uses -1 for reco failure
    """
    diag_rows = []
    for col in features:
        before = df[col].isna().sum()

        df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        df.loc[np.abs(df[col]) > 0.1 * float32_max, col] = np.nan
        df.loc[np.abs(df[col]) >= SENTINEL_THRESHOLD, col] = np.nan
        if col in SENTINEL_MINUS1_FEATURES:
            df.loc[df[col] == -1, col] = np.nan

        n_replaced = df[col].isna().sum() - before
        if n_replaced > 0:
            diag_rows.append({
                'feature':       col,
                'set_to_NaN':    n_replaced,
                'set_to_NaN_%':  f'{100 * n_replaced / len(df):.2f}',
            })
    return pd.DataFrame(diag_rows)


print("=" * 70)
print("🧹 CLEANING BDT TRAINING FEATURES  (sentinel → NaN)")
print("=" * 70)
print(f"   Sentinel threshold : |x| ≥ {SENTINEL_THRESHOLD}")
print(f"   Overflow threshold : |x| > {0.1 * float32_max:.2e}")
print(f"   -1 sentinel features: {SENTINEL_MINUS1_FEATURES}")
print(f"   Features to clean  : {len(training_parameters)}\n")

diag_train = replace_bad_with_nan(df_pre_train, training_parameters)
diag_test  = replace_bad_with_nan(df_pre_test,  training_parameters)
diag_full  = replace_bad_with_nan(df_pre,        training_parameters)

# Merge diagnostics for display
if diag_train.empty and diag_test.empty:
    print("✅ No bad values found — all feature columns are already clean.")
else:
    diag = (
        diag_train.rename(columns={'set_to_NaN': 'train_NaN', 'set_to_NaN_%': 'train_%'})
        .merge(
            diag_test.rename(columns={'set_to_NaN': 'test_NaN', 'set_to_NaN_%': 'test_%'}),
            on='feature', how='outer'
        )
        .fillna(0)
    )
    print(f"⚠️  Bad values → NaN in {len(diag)} / {len(training_parameters)} feature columns:\n")
    print(diag.to_string(index=False))

print("\n" + "=" * 70)
print("✅ Cleaning complete. NaN values will be handled natively by XGBoost.")
print("=" * 70)


In [ ]:
# ── BDT Hyperparameters ───────────────────────────────────────────────────────
# Define once here — passed into both the optimisation step and the final
# training so that round-count search and final model always use the same params.
# scale_pos_weight is NOT set here; it is computed from the data in both steps.
params = {
    'objective':        'binary:logistic',
    'booster':          'gbtree',
    'eta':              0.02,
    'tree_method':      'exact',
    'max_depth':        3,
    'subsample':        0.8,
    'colsample_bytree': 1,
    'verbosity':        0,
    'min_child_weight': 1,
    'seed':             2002,
    'gamma':            1,
    'max_delta_step':   0,
    'eval_metric':      ['error', 'auc', 'aucpr'],
}

print("✅ BDT hyperparameters defined:")
for k, v in params.items():
    print(f"   {k}: {v}")

In [ ]:
# Optimise number of boosting rounds with early stopping
print("\n🚀 Optimising number of boosting rounds with early stopping...\n")
print("This will train up to 1000 rounds with early_stopping_rounds=50")
print("and plot AUC/AUCPR curves to determine optimal performance.\n")

metrics_df = sf.bdt_metrics(
    df_pre_train,
    df_pre_test,
    TRAIN_QUERY,
    TEST_QUERY,
    training_parameters,
    ISRUN3,
    save=False,
    verbose=True,
    params=params,   # <-- same hyperparameters used in final training
)

print("\n✅ Review the AUC and AUCPR plots above to determine optimal number of rounds")
print("💡 Look for where the test curves plateau or start to diverge from train")
print("💡 The model uses early_stopping_rounds=50, so training stops 50 rounds after best score")

In [ ]:
metrics_df.to_pickle(f'bdt_metrics_df_{HORN_CURRENT}_today.pkl')
print(f"✅ Saved bdt_metrics_df to bdt_metrics_df_{HORN_CURRENT}_today.pkl")

## BDT Training - Step 2: Train Final Model

Now train the final model with the optimized number of rounds (set ROUNDS below based on the plots above).

In [ ]:
# Set the number of boosting rounds based on the optimization above
# Look at where the AUC/AUCPR curves plateau in the plots
ROUNDS = 250  # TODO: Update this based on the plots above!

print(f"🎯 Training final BDT model with {ROUNDS} boosting rounds...")

In [ ]:
# Reload selection_functions to pick up any changes
import importlib
import backend_functions.selection_functions as sf
importlib.reload(sf)

print("✅ Reloaded selection_functions module")


In [ ]:
# Train the BDT using the pre-prepared train/test sets from the optimization step
# (avoids redundant addRelevantColumns_flexible + train_test_split inside main_BDT)
print("\n🚀 Starting final BDT training...\n")

# scale_pos_weight is derived from the TRAINING set only (after TRAIN_QUERY),
# since that is the actual data XGBoost sees — not the combined train+test df_pre.
scale_pos_weight = (
    len(df_pre_train.query(TRAIN_QUERY + ' and is_signal == False')) /
    len(df_pre_train.query(TRAIN_QUERY + ' and is_signal == True'))
)
print(f"scale_pos_weight (background / signal): {scale_pos_weight:.4f}")

# Inject scale_pos_weight into a copy of the shared params dict
training_params = dict(params)
training_params['scale_pos_weight'] = scale_pos_weight

# df_pre, df_pre_train, df_pre_test are already prepared in the optimization step above
queried_test_df, bdt_model, bdt_results_df = sf.bdt_raw_results(
    df_pre_train, df_pre_test,
    TRAIN_QUERY, TEST_QUERY,
    training_parameters, training_params, ROUNDS
)

bdt_results = {
    'bdt_results_df': bdt_results_df,
    'queried_test_df': queried_test_df,
    'bdt_model': bdt_model,
    'df_pre_train': df_pre_train,
    'df_pre_test': df_pre_test,
    'df_pre': df_pre
}

print("\n✅ BDT training complete!")
print(f"   Model trained on {len(df_pre_train):,} events")
print(f"   Tested on {len(df_pre_test):,} events")

## Evaluate Model Performance

In [ ]:
# Look at training results
print("📊 BDT Training Results:")
print(f"   Training set size: {len(df_pre_train):,}")
print(f"   Test set size: {len(df_pre_test):,}")
print(f"   Signal events in training: {len(df_pre_train[df_pre_train['is_signal']==True]):,}")
print(f"   Background events in training: {len(df_pre_train[df_pre_train['is_signal']==False]):,}")

# Show some metrics from the results dataframe
if len(bdt_results_df) > 0:
    print("\n🎯 Model Performance (last 10 rounds):")
    print(bdt_results_df.tail(10))

### Comparison of whether the POT weighted training is a better choice or not

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# ── 1. Build weighted DMatrix ─────────────────────────────────────────────────
train_queried = df_pre_train.query(TRAIN_QUERY).copy()

# Same secondary overflow cleaning prep_sets does internally
X_tr = train_queried[training_parameters].copy()
for col in training_parameters:
    X_tr.loc[np.abs(X_tr[col]) > 1e37, col] = np.nan

y_tr = train_queried['is_signal'].astype(int)

# Normalize weights to mean=1 so the scale doesn't interact with eta
raw_w = train_queried['totweight_data'].fillna(1.0).clip(lower=0)
norm_w = raw_w / raw_w.mean()

dtrain_w = xgb.DMatrix(data=X_tr, label=y_tr, missing=np.nan, weight=norm_w)

# Build test DMatrix (same for both models)
test_queried = df_pre_test.query(TEST_QUERY).copy()
X_te = test_queried[training_parameters].copy()
for col in training_parameters:
    X_te.loc[np.abs(X_te[col]) > 1e37, col] = np.nan
y_te = test_queried['is_signal'].astype(int)
dtest = xgb.DMatrix(data=X_te, missing=np.nan)

# Also need train DMatrix without weights (same events, for fair evals_result)
dtrain_uw = xgb.DMatrix(data=X_tr, label=y_tr, missing=np.nan)
dtest_lbl  = xgb.DMatrix(data=X_te, label=y_te, missing=np.nan)

# ── 2. Train weighted model ───────────────────────────────────────────────────
spw_w = (y_tr == 0).sum() / (y_tr == 1).sum()
params_w = dict(training_params, scale_pos_weight=spw_w)

bdt_weighted = xgb.train(
    params_w,
    dtrain_w,
    num_boost_round=ROUNDS,
    evals=[(dtrain_w, 'train'), (dtest_lbl, 'test')],
    verbose_eval=False,
)

# ── 3. Get predictions ────────────────────────────────────────────────────────
scores_uw = bdt_model.predict(dtest)          # unweighted (already trained)
scores_w  = bdt_weighted.predict(dtest)       # weighted

# ── 4. Metrics ────────────────────────────────────────────────────────────────
print(f"{'Metric':<25} {'Unweighted':>12} {'Weighted':>12}")
print("-" * 51)
print(f"{'AUC (ROC)':<25} {roc_auc_score(y_te, scores_uw):>12.4f} {roc_auc_score(y_te, scores_w):>12.4f}")
print(f"{'AUCPR':<25} {average_precision_score(y_te, scores_uw):>12.4f} {average_precision_score(y_te, scores_w):>12.4f}")

# ── 5. Overlaid ROC curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (scores, label, color) in zip(
    [axes[0], axes[0]],
    [(scores_uw, 'Unweighted', 'steelblue'), (scores_w, 'Weighted', 'darkorange')]
):
    fpr, tpr, _ = roc_curve(y_te, scores)
    auc_val = roc_auc_score(y_te, scores)
    axes[0].plot(fpr, tpr, color=color, label=f'{label} (AUC={auc_val:.4f})')

axes[0].plot([0,1],[0,1],'k--', lw=0.8)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

# ── 6. BDT score distributions ────────────────────────────────────────────────
sig_mask = y_te.values == 1
bkg_mask = ~sig_mask
bins = np.linspace(0, 1, 50)

for scores, label, ax_col in [(scores_uw, 'Unweighted', 'steelblue'), (scores_w, 'Weighted', 'darkorange')]:
    axes[1].hist(scores[sig_mask], bins=bins, histtype='step', density=True,
                 label=f'{label} sig',  linestyle='-',  color=ax_col)
    axes[1].hist(scores[bkg_mask], bins=bins, histtype='step', density=True,
                 label=f'{label} bkg',  linestyle='--', color=ax_col)

axes[1].set_xlabel('BDT score'); axes[1].set_ylabel('Density (a.u.)')
axes[1].set_title('Score distributions'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Save Trained Model

Save the trained BDT model with a descriptive name.

In [ ]:
from pathlib import Path
import pickle

# Create output directories if needed
Path('BDT_models').mkdir(parents=True, exist_ok=True)
Path('BDT_training_data').mkdir(parents=True, exist_ok=True)

# Create model filename based on configuration
run_names = '_'.join([k for k, v in RUN_PERIODS.items() if v])
ext_suffix = '_noext' if not USE_EXT_IN_BDT else '_withext'
horn_suffix = f'_{HORN_CURRENT}'
model_filename = f'BDT_models/bdt_{run_names}{horn_suffix}{ext_suffix}.model'

# Save model
bdt_model.save_model(model_filename)
print(f"✅ Model saved to: {model_filename}")

# Save training parameters as well
training_info = {
    'model_path':        model_filename,
    'horn_current':      HORN_CURRENT,
    'training_variables': training_parameters,
    'run_periods':       RUN_PERIODS,
    'train_test_split':  TRAIN_TEST_SPLIT,
    'event_split_seed':  EVENT_SPLIT_SEED,
    'use_ext_in_bdt':    USE_EXT_IN_BDT,
    'train_query':       TRAIN_QUERY,
    'test_query':        TEST_QUERY,
    'rounds':            ROUNDS,
    'isrun3':            ISRUN3,
    'nue_intrinsic':     NUE_INTRINSIC,
    'params':            params,                                        # hyperparameters (without scale_pos_weight)
    'scale_pos_weight':  training_params['scale_pos_weight'],           # data-derived value used in training
}

info_filename = model_filename.replace('.model', '_info.pkl')
with open(info_filename, 'wb') as f:
    pickle.dump(training_info, f)

print(f"✅ Training info saved to: {info_filename}")
print(f"   Hyperparameters: {params}")
print(f"   scale_pos_weight: {training_params['scale_pos_weight']:.4f}")

## Save Training Data (Optional)

Save the training data for later analysis (SHAP, etc.)

In [ ]:
# Save training data
training_data = {
    'df_pre_train':  df_pre_train,
    'df_pre_test':   df_pre_test,
    'df_pre':        bdt_results['df_pre'],
    'training_info': training_info,
}

training_data_filename = f'BDT_training_data/training_data_{run_names}_multirun{ext_suffix}.pkl'

with open(training_data_filename, 'wb') as f:
    pickle.dump(training_data, f)

print(f"✅ Training data saved to: {training_data_filename}")
print(f"   Train: {len(df_pre_train):,} events")
print(f"   Test:  {len(df_pre_test):,} events")

## Summary

Training complete! The model is now ready to use in your selection analysis.

In [ ]:
print("=" * 70)
print("🎉 BDT TRAINING COMPLETE!")
print("=" * 70)
print(f"\n📁 Model saved to: {model_filename}")
print(f"📁 Training info saved to: {info_filename}")
print(f"📁 Training data saved to: {training_data_filename}")
print(f"\n📊 Training Summary:")
print(f"   Run periods: {[k for k, v in RUN_PERIODS.items() if v]}")
print(f"   Total training events: {len(df_pre_train):,}")
print(f"   Total test events: {len(df_pre_test):,}")
print(f"   Training variables: {len(training_parameters)}")
print(f"   Boosting rounds: {ROUNDS}")
print(f"\n💡 Next Steps:")
print(f"   1. Update top.py to point to: {model_filename}")
print(f"   2. Use selection.ipynb to analyze individual run periods")
print(f"   3. The same BDT model will be applied consistently to all runs")
print("=" * 70)